# Data Description
- Author: 詹明翰、謝子詮、曾翊涵；共同作者不等於已完成個別工作。
- Date first created: 20261127
- Background: Unit2.7機器可讀Feature Registry，記錄Final Annual／Pooled欄位來源、定義、型態、角色、時間／地理、缺失／零、readiness、用途與防洩漏規格；不是第三份分析CSV。
- Unit of observation: 一個Final欄位名稱；494個兩份Final欄位聯集條目，各有annual／pooled實際variant來源，兩份Final各490欄。
- Sample period: 事故及B1為2024–2025；B2/B3來源快照限制保留，不作年度獨立觀測。
- Known issues: D原版維持；主要分群研究候選骨架為S0 pooled456里，A1八項／A2九項尚未定案。S0固定保守、S1固定幾何唯一必要穩健性、S2歷史保守、S3歷史幾何唯一敏感度，不作獨立樣本。Annual3648鍵specification＋village_id＋year、pooled1824鍵specification＋village_id；同里跨年非獨立樣本。本單元保留全部骨架、零事故及低分母里，不決定正式樣本、低樣本門檻或模型。

A僅事故組成候選；B1設籍人口為核心外部比較，非交通暴露或涉事者住地；B2補充分區工作分類與條件式範圍，非完整分區／實際土地使用，五observed不重正規化、不用區間中點補未知。群間分區比較須同報I/A與Q/A，湖田I=0不代表分區確定不存在。B3道路兩項優先外部比較，號誌／行穿／計畫路寬有條件補充，MRT2026僅當期補充；原技術readiness與新研究用途分開。號誌A／B／LCS與MRT300／500／800m敏感度保留，Blocked沒有假0欄。C為分群後負擔，死亡時間窗／R20未解除，不當真正暴露校正風險或因果效果。

B1 annual12月、pooled24月，各自人口月和／分母保留，不用pooled替代年度人口。龍福／幸福2025-06人口跳動保留原值，新增品質flag／note；搬遷及暫遷制度背景不能證明直接統計原因，不重查。B2／B3固定快照只按village_id重複連接，不形成年度設施變化或認證事故前環境。未來2024特徵選擇、平滑／低樣本擬合參數、Scaler／Imputer／PCA／分群中心只能用2024擬合，不能用2025或pooled。MRT2026不得冒稱事前環境；環境、負擔、分母及品質不自動入主分群，不以select_dtypes挑所有數值欄。

同名欄只有明示共同語義且逐值／dtype／NA完全一致才合併；其他同名以family前綴保留，來源映射逐欄保存，不接受_x／_y。固定面積沿Unit2.2，不用原AREA，歷史密度分母只沿S2/S3原契約；原名稱糖?里／糖蔀里與標準糖廍里、provisional_match都保留，不重新查碼或設NA。公館2025四規格零事故仍有環境值，事故比例／均數NA；路寬三里無樣本為NA，不完整案例刪里。

微量GIS比例超界只讀來源值及分子分母核對，不重跑GIS。B2 185里／B3 93里原旗標保留，既有面積容忍0.1m²；本輪沒有後續運算需要嚴格0–1，因此不新增analysis_safe或clip。無補值、winsorize、log、標準化、平滑、正式EDA／相關／檢定／PCA／分群／預測／Finding。依賴已保存正式來源與純表格validator，不執行舊Notebook、事故前處理或大型GIS。完成停止Unit2.7，Chapter3只提出交接建議。
- Definition for each variable: 下方完整494條feature records與registry top-level欄位說明，schema variant逐一核對；未建立Blocked候選另記，不造數值欄。欄位source_dataset／source_column為一或多個來源，dataset_variants明確保存各資料集的定義／來源差異。
- Data path: `data/processed/taipei_feature_registry.json`
- Source data: INPUTS精確50項來源／helper綁定；input_manifest九份正式CSV版本，chapter2_inventory完整17項資料／支援／metadata交付。
- Processing code: [主處理Notebook](../code/prepare_taipei_final_analytical_datasets.ipynb)；[helper](../code/unit27_integration_support.py)。Producer SHA `b0083fc4b44c5193f95b09dc669fac9cf9744e1bc1364760d860872bc7a452ba`。
- Role: metadata_feature_use_contract。
- Verification status: 3,213,528 bytes；主流程兩次乾淨kernel五輸出相同。本文件獨立讀JSON、兩份Final CSV／schema並互相核對，無GIS或上游處理重跑。
- SHA-256: `57c36e5572be721de86dabe992be2a0e0828893bab493c6ae104884cd062309c`
- Downstream analysis input: `data/processed/taipei_feature_registry.json`；分析數值仍取兩份Final CSV，registry不提供第三份數值資料。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。預定分工不等於已完成貢獻。Actual Contribution：本輪Codex協助，三位共同作者維持；成員實際修改、DE交付確認、DA驗收、PM審查及本人Git提交待完成。

本Registry不內嵌自身hash以避免循環；自身大小／SHA由本獨立文件與PROJECT核對。schema可綁Registry SHA，Registry只記schema路徑；最終五輸出hash由本文件共同固定。


## Packages、精確來源及最終五輸出版本

In [1]:
from pathlib import Path
import sys,json,hashlib,platform
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():PROJECT_ROOT=p;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
INPUTS={'data/processed/taipei_village_base.csv': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'data/processed/taipei_village_base_schema.json': 'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070', 'data/processed/taipei_village_base_documentation.ipynb': 'a843d587788b4ea6a943cc05d9413908cf8b66b2f73f9fdbec01e2ab52edd535', 'data/processed/taipei_dataset_a_annual.csv': '4e264176c970263777eb4c564f427e1f839a45b31552a37bc4bdbfbed6b09422', 'data/processed/taipei_dataset_a_annual_schema.json': '4d0c5b7b307b1cca4261a9595cd1ca894f549d88e06bed1b0b071dcf7c40fb21', 'data/processed/taipei_dataset_a_annual_documentation.ipynb': 'a7713b7dd5edd26d31de5437547b4c987fc1992443b45c98eca5e385921b5be5', 'data/processed/taipei_dataset_a_pooled.csv': '081c7dd42c94e16a5684f27edaebd310a9c556bae7460783dd87fde8d26ea46a', 'data/processed/taipei_dataset_a_pooled_schema.json': '7878146fac39d4bfabee965b6001d0cf5e101ca1d76778599fec4b480ad59b5a', 'data/processed/taipei_dataset_a_pooled_documentation.ipynb': '212415a1e11fda62c44b36990a5a2e9c1713beb68f4e2c6879018e5685acf1da', 'data/processed/taipei_dataset_b1_annual.csv': 'd4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795', 'data/processed/taipei_dataset_b1_annual_schema.json': '89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca', 'data/processed/taipei_dataset_b1_annual_documentation.ipynb': '214afc23ff8773fb32384577aca1a499c00b4b1cccc9e77bdb2afe6d74ad84b6', 'data/processed/taipei_dataset_b1_pooled.csv': '3b02fb7b338b59f2552972ccb98efa0d723e525826b301cdda021efa9ddd4e50', 'data/processed/taipei_dataset_b1_pooled_schema.json': '63f46e1ae0cba9c19549ee5f8b9ad39084a22f5d2d738b62567b1b3f5e86f340', 'data/processed/taipei_dataset_b1_pooled_documentation.ipynb': '3f90cfdc8d34241fabb182886320aa243a88a9fc6dd9715d1c519dcb219fca34', 'data/processed/taipei_dataset_c_annual.csv': '9ab1970ed0f790ebdf31e1726094493305da00ac4efaf84b8a554f7debfa560e', 'data/processed/taipei_dataset_c_annual_schema.json': '4a8151a9a5dcd716cc03312b303c4455cb47210192c43b86cf3a0e06d268cb6f', 'data/processed/taipei_dataset_c_annual_documentation.ipynb': '32b8d1b015cbe2645bfd05a0af8f581692950d281c139c0e69b9b1f37286d8df', 'data/processed/taipei_dataset_c_pooled.csv': 'caceea61ff906b3e2f0b4796a31af3373a929307a0c7f7094fb9373794f76bc6', 'data/processed/taipei_dataset_c_pooled_schema.json': '44baac19975650d792e23e81c5463b84e00f299c9c606ef0f95abf8b3756d3ab', 'data/processed/taipei_dataset_c_pooled_documentation.ipynb': '0cc9c793ffbccf1cb03c5756319275e81dcbbcd211eae733d65bbd16575e7cb1', 'data/processed/taipei_dataset_b2.csv': '1af7ce133e62a1188e8755a1e4e7bc55db8b89d639f089f1c75319c8a9ee0269', 'data/processed/taipei_dataset_b2_schema.json': 'd5857362444df6bea1f606330d0363d792cc1a7f56812c9bb4917ce1b5e3a102', 'data/processed/taipei_dataset_b2_documentation.ipynb': '5d03694ab3e4e342e448b95bbcae99f767ff399a31f40a8d8cd2ff7b58af8ce0', 'data/processed/taipei_dataset_b3.csv': '36d6f42a86a54ba39552c611332c5185c89a3740e14f5e9e0268158ec8d9debe', 'data/processed/taipei_dataset_b3_schema.json': '440f6020e336990071a3492eba2f42ea36229f11948a8ea14f59427c9e25b8bf', 'data/processed/taipei_dataset_b3_documentation.ipynb': 'f7fc449cef4108bc1c7c89a1a39bcb51ba371b414848a857324be3d645b40962', 'data/processed/taipei_crashes_2024_2025_standardized.csv': 'b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252', 'data/processed/taipei_crashes_2024_2025_standardized_schema.json': 'c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650', 'data/processed/taipei_crash_spatial_assignment_2024_2025.csv': '2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5', 'data/processed/taipei_crash_spatial_assignment_2024_2025_schema.json': 'd43e0e3bcc1af2b7683d9466ccb4c441c39a6d9ce56f957b0ed5a89c1b487ad3', 'data/code/prepare_taipei_dataset_a_and_c.ipynb': '2557facf8a80ee38642b3e2220814627c588888a28536c9d613ab398dffe1421', 'data/code/prepare_taipei_dataset_b1.ipynb': '27ef88b179ae346479cab1d53b35c41a537cf7383d03216d4c4923c32eaa7ff0', 'data/code/prepare_taipei_dataset_b2.ipynb': 'd3666314a709587732dc23905837d3c62014cd3cc1b142c7221037dc21d1b00e', 'data/code/prepare_taipei_dataset_b3.ipynb': 'c10ccbd944f96710af4cbb25b7bb980bf80ef50e109ecbae8a40a863880c4f06', 'data/code/prepare_taipei_crashes_2024_2025.ipynb': 'c14dbd34bd3d776339e292b820a049b792a30ec49f2a8fc0173440ed6e146a9e', 'data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb': 'ef8a3d21f38ecb902f9495720b0094363ebd57cf444b837176fe2b815992abc4', 'data/code/unit22_spatial_support.py': '06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc', 'data/code/unit23_feature_support.py': '0d2078a764239aba8035355b224193c3b4c073995c860575bf4ed19834607fcc', 'data/code/unit24_population_support.py': 'ce49653ccfec3fc9000f56f58076021ac906764806780d0c7a9e44b4aa2060a4', 'data/code/unit25_zoning_support.py': 'a5d1a8bbc8b0473921340b01af09506902123030502a61035def77c83af7b2a4', 'data/code/unit26_transportation_support.py': 'dcdbeb261cec78d6eb3d967755da79607d25186671dea3988d61696db7f6c557', 'data/processed/village_crosswalk.csv': 'dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69', 'data/processed/village_crosswalk_documentation.ipynb': '768ebaac028b635c93b9c768efefdb504c902d49da586e5f33180a7c9855fce8', 'data/processed/urban_zoning_crosswalk.csv': '4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f', 'data/processed/urban_zoning_crosswalk_documentation.ipynb': 'b931add933d943890b9ec8fbed463836d28838632cec3a5f0636ff0f6b9fa36d', 'data/processed/urban_zoning_crosswalk_v2.csv': '3af29de364a10cd135252b9851ce6bbb74586ab47d5c5d68ade036e4a2960583', 'data/processed/urban_zoning_crosswalk_v2_schema.json': '5925bf3b77fd129bfd1e7eb4aaf765596b4d26fe0228b1c34f3a9935f9711a5f', 'data/processed/urban_zoning_crosswalk_v2_documentation.ipynb': 'f2e671c7470dbcb772dd4155bbd59ed101cc9ed7bf336c1fae7f6ddc23bcb48e', 'data/code/unit27_integration_support.py': '21367cd5b3bab12eab10c63cbf8e1d67b808b4fc4ba31e1ecc242daab0f7867d'}
PROJECT_CONTEXT={'path': 'PROJECT.md', 'before_unit27_sha256': '3beff393182dfb94c33dd22dda130ca4b0117ce2e2bb2758a742de814f3ad79b', 'role': 'reviewed mutable research context, authorized Unit2.7 updates; not numeric integration input'}
for rel,h in INPUTS.items():assert (PROJECT_ROOT/rel).is_file() and sha(PROJECT_ROOT/rel)==h,('INPUT HASH MISMATCH',rel)
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
import unit27_integration_support as u
print('Python',platform.python_version(),'正式來源／文件／helper綁定',len(INPUTS),'缺值契約 __MISSING__；無**MISSING**契約')

EXPECTED_OUTPUTS={'data/processed/taipei_feature_registry.json': '57c36e5572be721de86dabe992be2a0e0828893bab493c6ae104884cd062309c', 'data/processed/taipei_final_analytical_annual.csv': 'b2a6e14d945555b1fa3c4f01d7eadbe38cda4b537cd4327fb16e368fa29d74e6', 'data/processed/taipei_final_analytical_annual_schema.json': '06c3979a676d6124a4fa2db8d7e0cea8129ccec1211947d15750408ee952d8ca', 'data/processed/taipei_final_analytical_pooled.csv': 'ce31390a3af8d941e93b0f150f377f01772b7d748aa75756ef66c8d55a4022e2', 'data/processed/taipei_final_analytical_pooled_schema.json': '471c04ca171adc5d49e2a2647460b9d3e896acb1a5088ccccb86bfe5ea95227e'}
assert sha(PROJECT_ROOT/'data/code/prepare_taipei_final_analytical_datasets.ipynb')=='b0083fc4b44c5193f95b09dc669fac9cf9744e1bc1364760d860872bc7a452ba'
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h,('OUTPUT HASH MISMATCH',rel)
registry=json.loads((PROJECT_ROOT/u.REGISTRY).read_text(encoding='utf-8'))
print('只讀最終保存檔案／Registry／必要里主表，無上游處理、整合重做或GIS。')


Python 3.12.2 正式來源／文件／helper綁定 50 缺值契約 __MISSING__；無**MISSING**契約
只讀最終保存檔案／Registry／必要里主表，無上游處理、整合重做或GIS。


## Change for Your Own Data

JSON為UTF-8／LF，欄位角色與Final dtype按schema，不將metadata boolean／null混入數值缺值。

In [2]:
frames={};contracts={}
for period in ['annual','pooled']:
    sc=json.loads((PROJECT_ROOT/u.path(u.FINAL[period],'_schema.json')).read_text(encoding='utf-8'));contracts[period]=sc
    frames[period]=u.read_csv(PROJECT_ROOT/u.path(u.FINAL[period]),sc)
    u.validate_documented(PROJECT_ROOT,frames[period],sc,registry,period)
assert u.validate_registry(registry,frames,{p:contracts[p]['columns'] for p in frames})
features=pd.DataFrame(registry['features']);assert len(features)==494
top=pd.DataFrame([{'key':k,'type':type(v).__name__,'meaning':{'features':'每Final欄位的完整角色／定義／來源與各資料集variant','datasets':'Final各自欄名／主鍵／形狀／CSVhash／schema與說明路徑','input_manifest':'九正式CSV各版本／schema／觀察單位／時間／限制','source_inputs':'49固定既有檔＋本輪helper精確SHA','candidate_feature_sets':'A1/A2／環境／負擔／敏感度明確欄位集合','blocked_features':'未建立候選及理由，不含假0','temporal_isolation':'2024擬合與2025驗證／固定快照／非獨立樣本限制','chapter2_inventory':'17項完整正式／支援／metadata資料交付','structural_dependence_notes':'定義與共同分母關係，非本輪相關性分析'}.get(k,'研究角色、來源品質、數值政策或驗收／版本metadata')} for k,v in registry.items()])
u.show('Registry頂層欄位字典',top)


**Registry頂層欄位字典**

,key,type,meaning
0,unit,str,研究角色、來源品質、數值政策或驗收／版本metadata
1,created_at,str,研究角色、來源品質、數值政策或驗收／版本metadata
2,registry_version,str,研究角色、來源品質、數值政策或驗收／版本metadata
3,role,str,研究角色、來源品質、數值政策或驗收／版本metadata
4,producer,str,研究角色、來源品質、數值政策或驗收／版本metadata
5,helper,str,研究角色、來源品質、數值政策或驗收／版本metadata
6,documentation,str,研究角色、來源品質、數值政策或驗收／版本metadata
7,source_inputs,dict,49固定既有檔＋本輪helper精確SHA
8,input_manifest,list,九正式CSV各版本／schema／觀察單位／時間／限制
9,processing_environment,dict,研究角色、來源品質、數值政策或驗收／版本metadata


,key,type,meaning
18,blocked_features,list,未建立候選及理由，不含假0
19,source_readiness,dict,研究角色、來源品質、數值政策或驗收／版本metadata
20,population_geographic_comparability_note,str,研究角色、來源品質、數值政策或驗收／版本metadata
21,B2_uncertainty_audit,dict,研究角色、來源品質、數值政策或驗收／版本metadata
22,numerical_boundary_audit,list,研究角色、來源品質、數值政策或驗收／版本metadata
23,analysis_safe_policy,str,研究角色、來源品質、數值政策或驗收／版本metadata
24,temporal_isolation,dict,2024擬合與2025驗證／固定快照／非獨立樣本限制
25,low_count_policy,str,研究角色、來源品質、數值政策或驗收／版本metadata
26,chapter2_inventory,list,17項完整正式／支援／metadata資料交付
27,limitations,list,研究角色、來源品質、數值政策或驗收／版本metadata


## Definition for each variable

In [3]:
u.show('完整494個Final欄位條目',features[['column_name','dtype','definition','unit','source_dataset','source_column','analytical_role','readiness','allowed_for_main_clustering','allowed_for_external_comparison','sensitivity_group']],block=16)

**完整494個Final欄位條目**

,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
0,specification,string,S0主分析；S1必要保守程度敏感度；S2歷史界；S3共同敏感度,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv]",[specification],identifier,Ready with explicit assumptions,False,False,NaN
1,village_id,string,固定CPTVID共同10碼研究ID,identifier,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_b1_annual.csv, data/processed/taipei_dataset_b2.csv, data/processed/taipei_dataset_b3.csv, data/processed/taipei_village_base.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv, data/processed/taipei_dataset_b1_pooled.csv]",[village_id],identifier,Ready with explicit assumptions,False,False,NaN
2,year,Int64,事故年度,year,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_b1_annual.csv]",[year],identifier,Ready with explicit assumptions,False,False,NaN
3,period_start_year,Int64,本列觀測期間起年,year,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv]",[period_start_year],numerator_denominator,Ready with explicit assumptions,False,False,NaN
4,period_end_year,Int64,本列觀測期間迄年,year,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv]",[period_end_year],numerator_denominator,Ready with explicit assumptions,False,False,NaN
5,period_years,Int64,觀測期間年數,year,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_b1_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv, data/processed/taipei_dataset_b1_pooled.csv]",[period_years],numerator_denominator,Ready with explicit assumptions,False,False,NaN
6,period,string,年度或兩年合併期間標示,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv]",[period],metadata,Ready with explicit assumptions,False,False,NaN
7,district_name,string,固定研究行政區,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_b1_annual.csv, data/processed/taipei_dataset_b2.csv, data/processed/taipei_dataset_b3.csv, data/processed/taipei_village_base.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv, data/processed/taipei_dataset_b1_pooled.csv]",[district_name],identifier,Ready with explicit assumptions,False,False,NaN
8,standard_village_name,string,標準里名，糖廍里已接受,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_b1_annual.csv, data/processed/taipei_dataset_b2.csv, data/processed/taipei_dataset_b3.csv, data/processed/taipei_village_base.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv, data/processed/taipei_dataset_b1_pooled.csv]",[standard_village_name],identifier,Ready with explicit assumptions,False,False,NaN
9,boundary_version,string,空間配對採固定或對應年歷史界,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv]",[boundary_version],metadata,Ready with explicit assumptions,False,False,NaN


,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
16,count_lt_30,boolean,觀測事故數<30候選品質旗標；不排除任何里,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv]",[count_lt_30],quality_indicator,Ready with explicit assumptions,False,False,NaN
17,count_lt_50,boolean,觀測事故數<50候選品質旗標；不排除任何里,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_c_annual.csv, data/processed/taipei_dataset_a_pooled.csv, data/processed/taipei_dataset_c_pooled.csv]",[count_lt_50],quality_indicator,Ready with explicit assumptions,False,False,NaN
18,morning_peak_numerator,Int64,07:00≤時間<09:00；有效且符合條件事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_numerator],numerator_denominator,Ready with explicit assumptions,False,False,NaN
19,morning_peak_denominator,Int64,07:00≤時間<09:00；該特徵有效事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_denominator],numerator_denominator,Ready with explicit assumptions,False,False,NaN
20,morning_peak_false_count,Int64,07:00≤時間<09:00；有效且不符合事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_false_count],numerator_denominator,Ready with explicit assumptions,False,False,NaN
21,morning_peak_missing_count,Int64,07:00≤時間<09:00；該特徵缺失或無效事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_missing_count],quality_indicator,Ready with explicit assumptions,False,False,NaN
22,morning_peak_coverage,Float64,07:00≤時間<09:00；有效數／觀測數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_coverage],quality_indicator,Ready with explicit assumptions,False,False,NaN
23,morning_peak_ratio,Float64,07:00≤時間<09:00；符合數／各自有效數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_ratio],clustering_candidate,Ready with explicit assumptions,True,False,NaN
24,morning_peak_denominator_lt_5,boolean,07:00≤時間<09:00有效分母<5候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_denominator_lt_5],quality_indicator,Ready with explicit assumptions,False,False,NaN
25,morning_peak_denominator_lt_10,boolean,07:00≤時間<09:00有效分母<10候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[morning_peak_denominator_lt_10],quality_indicator,Ready with explicit assumptions,False,False,NaN


,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
32,evening_peak_missing_count,Int64,17:00≤時間<19:00；該特徵缺失或無效事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_missing_count],quality_indicator,Ready with explicit assumptions,False,False,NaN
33,evening_peak_coverage,Float64,17:00≤時間<19:00；有效數／觀測數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_coverage],quality_indicator,Ready with explicit assumptions,False,False,NaN
34,evening_peak_ratio,Float64,17:00≤時間<19:00；符合數／各自有效數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_ratio],clustering_candidate,Ready with explicit assumptions,True,False,NaN
35,evening_peak_denominator_lt_5,boolean,17:00≤時間<19:00有效分母<5候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_denominator_lt_5],quality_indicator,Ready with explicit assumptions,False,False,NaN
36,evening_peak_denominator_lt_10,boolean,17:00≤時間<19:00有效分母<10候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_denominator_lt_10],quality_indicator,Ready with explicit assumptions,False,False,NaN
37,evening_peak_denominator_lt_20,boolean,17:00≤時間<19:00有效分母<20候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_denominator_lt_20],quality_indicator,Ready with explicit assumptions,False,False,NaN
38,evening_peak_denominator_lt_30,boolean,17:00≤時間<19:00有效分母<30候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_denominator_lt_30],quality_indicator,Ready with explicit assumptions,False,False,NaN
39,evening_peak_denominator_lt_50,boolean,17:00≤時間<19:00有效分母<50候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[evening_peak_denominator_lt_50],quality_indicator,Ready with explicit assumptions,False,False,NaN
40,late_night_numerator,Int64,時間≥22:00或<06:00；有效且符合條件事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[late_night_numerator],numerator_denominator,Ready with explicit assumptions,False,False,NaN
41,late_night_denominator,Int64,時間≥22:00或<06:00；該特徵有效事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[late_night_denominator],numerator_denominator,Ready with explicit assumptions,False,False,NaN


,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
48,late_night_denominator_lt_20,boolean,時間≥22:00或<06:00有效分母<20候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[late_night_denominator_lt_20],quality_indicator,Ready with explicit assumptions,False,False,NaN
49,late_night_denominator_lt_30,boolean,時間≥22:00或<06:00有效分母<30候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[late_night_denominator_lt_30],quality_indicator,Ready with explicit assumptions,False,False,NaN
50,late_night_denominator_lt_50,boolean,時間≥22:00或<06:00有效分母<50候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[late_night_denominator_lt_50],quality_indicator,Ready with explicit assumptions,False,False,NaN
51,weekend_numerator,Int64,星期六或日，非全部國定假日；有效且符合條件事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[weekend_numerator],numerator_denominator,Ready with explicit assumptions,False,False,NaN
52,weekend_denominator,Int64,星期六或日，非全部國定假日；該特徵有效事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[weekend_denominator],numerator_denominator,Ready with explicit assumptions,False,False,NaN
53,weekend_false_count,Int64,星期六或日，非全部國定假日；有效且不符合事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[weekend_false_count],numerator_denominator,Ready with explicit assumptions,False,False,NaN
54,weekend_missing_count,Int64,星期六或日，非全部國定假日；該特徵缺失或無效事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[weekend_missing_count],quality_indicator,Ready with explicit assumptions,False,False,NaN
55,weekend_coverage,Float64,星期六或日，非全部國定假日；有效數／觀測數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[weekend_coverage],quality_indicator,Ready with explicit assumptions,False,False,NaN
56,weekend_ratio,Float64,星期六或日，非全部國定假日；符合數／各自有效數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[weekend_ratio],clustering_candidate,Ready with explicit assumptions,True,False,NaN
57,weekend_denominator_lt_5,boolean,星期六或日，非全部國定假日有效分母<5候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[weekend_denominator_lt_5],quality_indicator,Ready with explicit assumptions,False,False,NaN


,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
64,intersection_false_count,Int64,位置代碼1–4，路口及附近／待轉／停等；有效且不符合事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_false_count],numerator_denominator,Ready with explicit assumptions,False,False,NaN
65,intersection_missing_count,Int64,位置代碼1–4，路口及附近／待轉／停等；該特徵缺失或無效事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_missing_count],quality_indicator,Ready with explicit assumptions,False,False,NaN
66,intersection_coverage,Float64,位置代碼1–4，路口及附近／待轉／停等；有效數／觀測數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_coverage],quality_indicator,Ready with explicit assumptions,False,False,NaN
67,intersection_ratio,Float64,位置代碼1–4，路口及附近／待轉／停等；符合數／各自有效數,fraction,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_ratio],clustering_candidate,Ready with explicit assumptions,True,False,NaN
68,intersection_denominator_lt_5,boolean,位置代碼1–4，路口及附近／待轉／停等有效分母<5候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_denominator_lt_5],quality_indicator,Ready with explicit assumptions,False,False,NaN
69,intersection_denominator_lt_10,boolean,位置代碼1–4，路口及附近／待轉／停等有效分母<10候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_denominator_lt_10],quality_indicator,Ready with explicit assumptions,False,False,NaN
70,intersection_denominator_lt_20,boolean,位置代碼1–4，路口及附近／待轉／停等有效分母<20候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_denominator_lt_20],quality_indicator,Ready with explicit assumptions,False,False,NaN
71,intersection_denominator_lt_30,boolean,位置代碼1–4，路口及附近／待轉／停等有效分母<30候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_denominator_lt_30],quality_indicator,Ready with explicit assumptions,False,False,NaN
72,intersection_denominator_lt_50,boolean,位置代碼1–4，路口及附近／待轉／停等有效分母<50候選旗標，不作正式排除,not applicable,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[intersection_denominator_lt_50],quality_indicator,Ready with explicit assumptions,False,False,NaN
73,rear_end_numerator,Int64,型態代碼13追撞；有效且符合條件事故數,accident,"[data/processed/taipei_dataset_a_annual.csv, data/processed/taipei_dataset_a_pooled.csv]",[rear_end_numerator],numerator_denominator,Ready with explicit assumptions,False,False,NaN


... (22 outputs left out)

,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
432,osm_major_road_ratio_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[osm_major_road_ratio_value_state],quality_indicator,Ready with explicit assumptions,False,False,NaN
433,observed_signal_location_density_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,[data/processed/taipei_dataset_b3.csv],[observed_signal_location_density_readiness_status],quality_indicator,Partially ready,False,False,NaN
434,observed_signal_location_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[observed_signal_location_density_value_state],quality_indicator,Partially ready,False,False,NaN
435,crosswalk_area_ratio_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,[data/processed/taipei_dataset_b3.csv],[crosswalk_area_ratio_readiness_status],quality_indicator,Partially ready,False,False,NaN
436,crosswalk_area_ratio_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[crosswalk_area_ratio_value_state],quality_indicator,Partially ready,False,False,NaN
437,median_annotated_road_width_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,[data/processed/taipei_dataset_b3.csv],[median_annotated_road_width_readiness_status],quality_indicator,Partially ready,False,False,NaN
438,median_annotated_road_width_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[median_annotated_road_width_value_state],quality_indicator,Partially ready,False,False,NaN
439,mrt_entrance_density_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,[data/processed/taipei_dataset_b3.csv],[mrt_entrance_density_readiness_status],quality_indicator,Ready with explicit assumptions,False,False,NaN
440,mrt_entrance_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[mrt_entrance_density_value_state],quality_indicator,Ready with explicit assumptions,False,False,NaN
441,mrt_entrance_buffer_500m_ratio_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,[data/processed/taipei_dataset_b3.csv],[mrt_entrance_buffer_500m_ratio_readiness_status],quality_indicator,Ready with explicit assumptions,False,False,NaN


,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
448,signal_geometry_only_location_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[signal_geometry_only_location_density_value_state],quality_indicator,Partially ready,False,False,NaN
449,signal_main_without_lcs_location_density_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,[data/processed/taipei_dataset_b3.csv],[signal_main_without_lcs_location_density_readiness_status],quality_indicator,Partially ready,False,False,NaN
450,signal_main_without_lcs_location_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[signal_main_without_lcs_location_density_value_state],quality_indicator,Partially ready,False,False,NaN
451,signal_geometry_without_lcs_location_density_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,[data/processed/taipei_dataset_b3.csv],[signal_geometry_without_lcs_location_density_readiness_status],quality_indicator,Partially ready,False,False,NaN
452,signal_geometry_without_lcs_location_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,[data/processed/taipei_dataset_b3.csv],[signal_geometry_without_lcs_location_density_value_state],quality_indicator,Partially ready,False,False,NaN
453,any_spatial_processing_failed,boolean,任一空間處理實際失敗；每個特徵另有value_state，不一概把所有特徵設NA,not applicable,[data/processed/taipei_dataset_b3.csv],[any_spatial_processing_failed],quality_indicator,Ready with explicit assumptions,False,False,NaN
454,any_core_feature_missing,boolean,七核心任一缺失，不刪除該行政里,not applicable,[data/processed/taipei_dataset_b3.csv],[any_core_feature_missing],quality_indicator,Ready with explicit assumptions,False,False,NaN
455,source_complete_census_certified,boolean,所有快照均不冒充同時點完整設施普查,not applicable,[data/processed/taipei_dataset_b3.csv],[source_complete_census_certified],quality_indicator,Ready with explicit assumptions,False,False,NaN
456,district_code_source,string,原固定SHP七碼CPTID；不是原事故區序數字碼，未截取或推定順序,not applicable,[data/processed/taipei_village_base.csv],[district_code_source],identifier,Ready with explicit assumptions,False,False,NaN
457,source_AREA_attribute,Float64,原AREA屬性僅追溯；單位／產製口徑未由本次認證，不作主要分母,原欄單位未認證,[data/processed/taipei_village_base.csv],[source_AREA_attribute],metadata,Ready with explicit assumptions,False,False,NaN


,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
464,fixed_geometry_wkb_sha256,string,本次讀取原固定geometry WKB摘要，供版本／不變性核對,not applicable,[data/processed/taipei_village_base.csv],[fixed_geometry_wkb_sha256],metadata,Ready with explicit assumptions,False,False,NaN
465,historical_boundary_code_2024,string,歷史原11碼VILLCODE，經既有人口碼對照共同10碼ID，非截碼,not applicable,[data/processed/taipei_village_base.csv],[historical_boundary_code_2024],identifier,Ready with explicit assumptions,False,False,NaN
466,historical_district_code_2024,string,歷史原TOWNCODE，與固定CPTID分開保留,not applicable,[data/processed/taipei_village_base.csv],[historical_district_code_2024],identifier,Ready with explicit assumptions,False,False,NaN
467,historical_village_name_2024,string,歷史原里名；年版快照不宣稱全年每天有效,not applicable,[data/processed/taipei_village_base.csv],[historical_village_name_2024],metadata,Ready with explicit assumptions,False,False,NaN
468,historical_area_m2_2024,Float64,該年版原面由3824轉3826後面積，歷史地理密度用此分母,m²,[data/processed/taipei_village_base.csv],[historical_area_m2_2024],numerator_denominator,Ready with explicit assumptions,False,False,NaN
469,historical_area_km2_2024,Float64,該歷史年版投影面積／1e6，不冒充固定共同分母,km²,[data/processed/taipei_village_base.csv],[historical_area_km2_2024],numerator_denominator,Ready with explicit assumptions,False,False,NaN
470,historical_match_status_2024,string,沿已接受crosswalk完整一對一，糖廍既定接受且provisional原狀態不變,not applicable,[data/processed/taipei_village_base.csv],[historical_match_status_2024],metadata,Ready with explicit assumptions,False,False,NaN
471,historical_source_crs_2024,string,歷史原始PRJ支持3824；計量轉至3826,not applicable,[data/processed/taipei_village_base.csv],[historical_source_crs_2024],metadata,Ready with explicit assumptions,False,False,NaN
472,historical_fixed_iou_2024,Float64,同研究里歷史與固定面之交集／聯集；不認定法定邊界變更,0–1,[data/processed/taipei_village_base.csv],[historical_fixed_iou_2024],numerator_denominator,Ready with explicit assumptions,False,False,NaN
473,historical_fixed_symdiff_m2_2024,Float64,同研究里歷史與固定面之對稱差面積,m²,[data/processed/taipei_village_base.csv],[historical_fixed_symdiff_m2_2024],numerator_denominator,Ready with explicit assumptions,False,False,NaN


,column_name,dtype,definition,unit,source_dataset,source_column,analytical_role,readiness,allowed_for_main_clustering,allowed_for_external_comparison,sensitivity_group
480,historical_match_status_2025,string,沿已接受crosswalk完整一對一，糖廍既定接受且provisional原狀態不變,not applicable,[data/processed/taipei_village_base.csv],[historical_match_status_2025],metadata,Ready with explicit assumptions,False,False,NaN
481,historical_source_crs_2025,string,歷史原始PRJ支持3824；計量轉至3826,not applicable,[data/processed/taipei_village_base.csv],[historical_source_crs_2025],metadata,Ready with explicit assumptions,False,False,NaN
482,historical_fixed_iou_2025,Float64,同研究里歷史與固定面之交集／聯集；不認定法定邊界變更,0–1,[data/processed/taipei_village_base.csv],[historical_fixed_iou_2025],numerator_denominator,Ready with explicit assumptions,False,False,NaN
483,historical_fixed_symdiff_m2_2025,Float64,同研究里歷史與固定面之對稱差面積,m²,[data/processed/taipei_village_base.csv],[historical_fixed_symdiff_m2_2025],numerator_denominator,Ready with explicit assumptions,False,False,NaN
484,historical_geometry_wkb_sha256_2025,string,轉3826後歷史geometry WKB摘要，沒有修復或替換原檔,not applicable,[data/processed/taipei_village_base.csv],[historical_geometry_wkb_sha256_2025],metadata,Ready with explicit assumptions,False,False,NaN
485,population_geographic_comparability_flag,boolean,兩個既有2025-06人口存量跳動里；不是排除規則,not applicable,[derived in Unit2.7],[district_name + standard_village_name],quality_indicator,Ready with explicit assumptions,False,False,NaN
486,population_geographic_comparability_note,string,既有事件證據界線與後續敏感度註記,not applicable,[derived in Unit2.7],[district_name + standard_village_name],metadata,Ready with explicit assumptions,False,False,NaN
487,b2_static_snapshot_repeated,boolean,B2按village_id重複連接固定快照，不代表年度變動,not applicable,[derived in Unit2.7],[upstream B2/B3 snapshot metadata],quality_indicator,Ready with explicit assumptions,False,False,NaN
488,b3_static_snapshot_repeated,boolean,B3按village_id重複連接混合來源固定快照，不代表年度變動,not applicable,[derived in Unit2.7],[upstream B2/B3 snapshot metadata],quality_indicator,Ready with explicit assumptions,False,False,NaN
489,environment_snapshot_not_verified_pre_accident,boolean,B2/B3不自動視為事前環境；尤其MRT2026,not applicable,[derived in Unit2.7],[upstream B2/B3 snapshot metadata],quality_indicator,Ready with explicit assumptions,False,False,NaN


## Summary

### Sample

### Summary Stats

In [4]:
u.show('實際前十個feature records',features[['column_name','dtype','analytical_role','recommended_use']].head(10))
u.show('實際角色統計',features.analytical_role.value_counts().rename_axis('role').reset_index(name='count'))
u.show('實際型態統計',features.dtype.value_counts().rename_axis('dtype').reset_index(name='count'))
u.show('候選集合及欄位',pd.DataFrame([{'set':k,'columns':v} for k,v in registry['candidate_feature_sets'].items()]))
u.show('定義／共同分母依賴，未作正式相關分析',pd.DataFrame(registry['structural_dependence_notes']))

**實際前十個feature records**

,column_name,dtype,analytical_role,recommended_use
0,specification,string,identifier,retain provenance; do not automatically model
1,village_id,string,identifier,retain provenance; do not automatically model
2,year,Int64,identifier,retain provenance; do not automatically model
3,period_start_year,Int64,numerator_denominator,preserve counts / areas / samples / denominators; no automatic clustering
4,period_end_year,Int64,numerator_denominator,preserve counts / areas / samples / denominators; no automatic clustering
5,period_years,Int64,numerator_denominator,preserve counts / areas / samples / denominators; no automatic clustering
6,period,string,metadata,retain provenance; do not automatically model
7,district_name,string,identifier,retain provenance; do not automatically model
8,standard_village_name,string,identifier,retain provenance; do not automatically model
9,boundary_version,string,metadata,retain provenance; do not automatically model


**實際角色統計**

,role,count
0,numerator_denominator,195
1,quality_indicator,174
2,metadata,40
3,sensitivity,39
4,identifier,13
5,supplementary_external,10
6,burden_outcome,9
7,clustering_candidate,9
8,core_external,3
9,priority_external,2


**實際型態統計**

,dtype,count
0,Float64,186
1,Int64,122
2,boolean,101
3,string,85


**候選集合及欄位**

,set,columns
0,A1,"[morning_peak_ratio, evening_peak_ratio, late_night_ratio, weekend_ratio, intersection_ratio, rear_end_ratio, side_collision_ratio, pedestrian_collision_ratio]"
1,A2,"[morning_peak_ratio, evening_peak_ratio, late_night_ratio, weekend_ratio, intersection_ratio, rear_end_ratio, side_collision_ratio, pedestrian_collision_ratio, multi_injury_ratio]"
2,strict_intersection_alternative,[strict_intersection_ratio]
3,B1_core_external,"[population_density, elderly_ratio, children_ratio]"
4,B2_supplementary_observed,"[residential_observed_share, commercial_observed_share, industrial_observed_share, educational_observed_share, park_green_observed_share]"
5,B3_priority_external,"[osm_road_density, osm_major_road_ratio]"
6,B3_conditional_supplementary,"[observed_signal_location_density, crosswalk_area_ratio, median_annotated_road_width]"
7,B3_2026_current_supplementary,"[mrt_entrance_buffer_500m_ratio, mrt_entrance_density]"
8,B3_sensitivity,"[mrt_entrance_buffer_300m_ratio, mrt_entrance_buffer_800m_ratio, signal_geometry_only_location_density, signal_main_without_lcs_location_density, signal_geometry_without_lcs_location_density]"
9,C_burden_and_sensitivity,"[observed_accident_count, injuries_observed_sum, fatalities_source_observed_sum, fatalities_2_30d_observed_sum, mean_injuries_per_accident, source_positive_fatality_ratio, r20_excluded_injuries_observed_sum, r20_excluded_fatalities_source_observed_sum, r20_excluded_fatalities_2_30d_observed_sum, mean_injuries_r20_difference, observed_accidents_per_fixed_km2, mean_annual_observed_accidents_per_fixed_km2, historical_mean_annual_accidents_per_km2]"


**定義／共同分母依賴，未作正式相關分析**

,features,note
0,"[morning_peak_ratio, evening_peak_ratio, late_night_ratio]",共同事故時間來源，定義依賴不是本輪相關性估計
1,"[rear_end_ratio, side_collision_ratio, pedestrian_collision_ratio]",共同原始碰撞型態欄，非獨立測量
2,"[morning_peak_ratio, evening_peak_ratio, late_night_ratio, weekend_ratio, intersection_ratio, rear_end_ratio, side_collision_ratio, pedestrian_collision_ratio, multi_injury_ratio]",九比例可重疊、分母各自有效，不是總和必為1的組成向量
3,"[multi_injury_ratio, mean_injuries_per_accident]",與C受傷負擔有定義關聯，不據此自動改權重或刪欄
4,"[population_density, osm_road_density, observed_signal_location_density, mrt_entrance_density]",共同固定面積分母可能形成結構依賴，不是實證相關性結果
5,"[osm_major_road_ratio, industrial_observed_share, mrt_entrance_density]",大量觀測零，不當缺失／現實不存在的自動證明
6,"[mrt_entrance_buffer_300m_ratio, mrt_entrance_buffer_500m_ratio, mrt_entrance_buffer_800m_ratio]",巢狀半徑具結構相關，非三項獨立設施觀測
7,"[residential_conditional_upper_bound, commercial_conditional_upper_bound, industrial_conditional_upper_bound, educational_conditional_upper_bound, park_green_conditional_upper_bound]",共用Q不確定聯集；各上界不可同時實現，不用中點補未知


### 版本依賴、交付清單與品質限制

In [5]:
u.show('Chapter2完整資料／支援／metadata交付',pd.DataFrame(registry['chapter2_inventory']))
u.show('九正式來源manifest',pd.DataFrame(registry['input_manifest']).drop(columns=['limitations']))
u.show('B3技術readiness',pd.DataFrame(registry['source_readiness']['B3']))
u.show('Blocked未建立候選',pd.DataFrame(registry['blocked_features']))
print(registry['temporal_isolation']);print(registry['analysis_safe_policy']);print(registry['population_geographic_comparability_note'])

**Chapter2完整資料／支援／metadata交付**

,dataset,role,path,rows,columns,bytes,sha256,schema_path,schema_sha256,producer,documentation,observation_unit,version,quality_status,entries,self_hash_policy
0,village_crosswalk,support_crosswalk,data/processed/village_crosswalk.csv,456.0,11.0,362330.0,dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69,NaN,NaN,data/code/taipei_population_validation.ipynb,data/processed/village_crosswalk_documentation.ipynb,one crosswalk key,"frozen existing file, no upstream rerun","local technical validation; original limitations retained, human acceptance pending",NaN,NaN
1,urban_zoning_crosswalk,support_classification_v1,data/processed/urban_zoning_crosswalk.csv,296.0,12.0,117968.0,4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f,NaN,NaN,data/code/taipei_urban_planning_geometry_validation.ipynb,data/processed/urban_zoning_crosswalk_documentation.ipynb,one crosswalk key,"frozen existing file, no upstream rerun","local technical validation; original limitations retained, human acceptance pending",NaN,NaN
2,urban_zoning_crosswalk_v2,support_classification_v2,data/processed/urban_zoning_crosswalk_v2.csv,296.0,21.0,139463.0,3af29de364a10cd135252b9851ce6bbb74586ab47d5c5d68ade036e4a2960583,data/processed/urban_zoning_crosswalk_v2_schema.json,5925bf3b77fd129bfd1e7eb4aaf765596b4d26fe0228b1c34f3a9935f9711a5f,data/code/prepare_taipei_dataset_b2.ipynb,data/processed/urban_zoning_crosswalk_v2_documentation.ipynb,five-field source compound classification key,"frozen existing file, no upstream rerun","local technical validation; original limitations retained, human acceptance pending",NaN,NaN
3,taipei_crashes_2024_2025_standardized,accident_level_source,data/processed/taipei_crashes_2024_2025_standardized.csv,45130.0,191.0,50400834.0,b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252,data/processed/taipei_crashes_2024_2025_standardized_schema.json,c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650,data/code/prepare_taipei_crashes_2024_2025.ipynb,data/processed/taipei_crashes_2024_2025_standardized_documentation.ipynb,"first-party operational candidate accident, not certified official unique event","frozen existing file, no upstream rerun","local technical validation; original limitations retained, human acceptance pending",NaN,NaN
4,taipei_village_base,geographic_base,data/processed/taipei_village_base.csv,456.0,41.0,345137.0,db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209,data/processed/taipei_village_base_schema.json,a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070,data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb,data/processed/taipei_village_base_documentation.ipynb,one fixed research village,"frozen existing file, no upstream rerun","local technical validation; original limitations retained, human acceptance pending",NaN,NaN
5,taipei_crash_spatial_assignment_2024_2025,accident_assignment_source,data/processed/taipei_crash_spatial_assignment_2024_2025.csv,45130.0,91.0,40948727.0,2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5,data/processed/taipei_crash_spatial_assignment_2024_2025_schema.json,d43e0e3bcc1af2b7683d9466ccb4c441c39a6d9ce56f957b0ed5a89c1b487ad3,data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb,data/processed/taipei_crash_spatial_assignment_2024_2025_documentation.ipynb,"one original Unit2.1 candidate accident, all retained","frozen existing file, no upstream rerun","local technical validation; original limitations retained, human acceptance pending",NaN,NaN
6,taipei_dataset_a_annual,formal_upstream_feature,data/processed/taipei_dataset_a_annual.csv,3648.0,128.0,2605036.0,4e264176c970263777eb4c564f427e1f839a45b31552a37bc4bdbfbed6b09422,data/processed/taipei_dataset_a_annual_schema.json,4d0c5b7b307b1cca4261a9595cd1ca894f549d88e06bed1b0b071dcf7c40fb21,data/code/prepare_taipei_dataset_a_and_c.ipynb,data/processed/taipei_dataset_a_annual_documentation.ipynb,specification × fixed research vil

**九正式來源manifest**

,dataset_family,dataset_version,path,rows,columns,bytes,sha256,schema_path,schema_sha256,documentation,documentation_sha256,primary_key,observation_unit,temporal_reference,producer
0,base,2.2 frozen saved file,data/processed/taipei_village_base.csv,456,41,345137,db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209,data/processed/taipei_village_base_schema.json,a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070,data/processed/taipei_village_base_documentation.ipynb,a843d587788b4ea6a943cc05d9413908cf8b66b2f73f9fdbec01e2ab52edd535,[village_id],one fixed research village,fixed research geography; separate historical 2024/2025 snapshots; not proof of monthly boundaries,data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb
1,a_annual,2.3 frozen saved file,data/processed/taipei_dataset_a_annual.csv,3648,128,2605036,4e264176c970263777eb4c564f427e1f839a45b31552a37bc4bdbfbed6b09422,data/processed/taipei_dataset_a_annual_schema.json,4d0c5b7b307b1cca4261a9595cd1ca894f549d88e06bed1b0b071dcf7c40fb21,data/processed/taipei_dataset_a_annual_documentation.ipynb,a7713b7dd5edd26d31de5437547b4c987fc1992443b45c98eca5e385921b5be5,"[specification, village_id, year]",specification × fixed research village × year,2024/2025 separately; 12 months for B1,data/code/prepare_taipei_dataset_a_and_c.ipynb
2,a_pooled,2.3 frozen saved file,data/processed/taipei_dataset_a_pooled.csv,1824,127,1362574,081c7dd42c94e16a5684f27edaebd310a9c556bae7460783dd87fde8d26ea46a,data/processed/taipei_dataset_a_pooled_schema.json,7878146fac39d4bfabee965b6001d0cf5e101ca1d76778599fec4b480ad59b5a,data/processed/taipei_dataset_a_pooled_documentation.ipynb,212415a1e11fda62c44b36990a5a2e9c1713beb68f4e2c6879018e5685acf1da,"[specification, village_id]",specification × fixed research village × pooled 2024–2025 period,"pooled2024–2025; 24 population months, never mean annual ratios",data/code/prepare_taipei_dataset_a_and_c.ipynb
3,b1_annual,2.4 frozen saved file,data/processed/taipei_dataset_b1_annual.csv,912,46,358874,d4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795,data/processed/taipei_dataset_b1_annual_schema.json,89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca,data/processed/taipei_dataset_b1_annual_documentation.ipynb,214afc23ff8773fb32384577aca1a499c00b4b1cccc9e77bdb2afe6d74ad84b6,"[village_id, year]",fixed research village × year,2024/2025 separately; 12 months for B1,data/code/prepare_taipei_dataset_b1.ipynb
4,b1_pooled,2.4 frozen saved file,data/processed/taipei_dataset_b1_pooled.csv,456,46,185548,3b02fb7b338b59f2552972ccb98efa0d723e525826b301cdda021efa9ddd4e50,data/processed/taipei_dataset_b1_pooled_schema.json,63f46e1ae0cba9c19549ee5f8b9ad39084a22f5d2d738b62567b1b3f5e86f340,data/processed/taipei_dataset_b1_pooled_documentation.ipynb,3f90cfdc8d34241fabb182886320aa243a88a9fc6dd9715d1c519dcb219fca34,[village_id],fixed research village × pooled 2024–2025 period,"pooled2024–2025; 24 population months, never mean annual ratios",data/code/prepare_taipei_dataset_b1.ipynb
5,c_annual,2.3 frozen saved file,data/processed/taipei_dataset_c_annual.csv,3648,80,1545763,9ab1970ed0f790ebdf31e1726094493305da00ac4efaf84b8a554f7debfa560e,data/processed/taipei_dataset_c_annual_schema.json,4a8151a9a5dcd716cc03312b303c4455cb47210192c43b86cf3a0e06d268cb6f,data/processed/taipei_dataset_c_annual_documentation.ipynb,32b8d1b015cbe2645bfd05a0af8f581692950d281c139c0e69b9b1f37286d8df,"[specification, village_id, year]",specification × fixed research village × year,2024/2025 separately; 12 months for B1,data/code/prepare_taipei_dataset_a_and_c.ipynb
6,c_pooled,2.3 frozen saved file,data/processed/taipei_dataset_c_pooled.csv,1824,79,799355,caceea61ff906b3e2f0b4796a31af3373a929307a0c7f7094fb9373794f76bc6,data/processed/taipei_dataset_c_pooled_schema.json,44baac19975650d792e23e81c5463b84e00f299c9c606ef0f95abf8b3756d3ab,data/processed/taipei_dataset_c_pooled_documentation.ipynb,0cc9c793ffbccf1cb03c5756319275e81dcbbcd211eae733d65bbd16575e

**B3技術readiness**

,feature,source,source_sha256,observation_unit,numerator,denominator,raw_crs,working_crs,crs_evidence,time_reference,spatial_completeness,readiness_status,recommended_analytical_use,key_limitations,human_acceptance,valid_N,missing_N,zero_N,min,q25,median,q75,max,mean,std_ddof1,value_state_counts,spatial_processing_valid_values,formal_analysis_selected
0,osm_road_density,data/raw/taiwan-250101-free.shp.zip,ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab,specified motor-candidate line representation,same-class-independent all included line union km excluding shared boundary,fixed_village_area_km2,EPSG:4326,EPSG:3826,A metadata,包內README明示2025-01-01T21:21:00Z快照；非2024／2025逐年獨立路網,source coverage not certified as a full census; frozen original villages preserved,Ready with explicit assumptions,priority external comparison candidate,"not centerline / traffic / VKT; fixed OSM snapshot, 2D union, class and boundary assumptions",DE / DA / PM pending,456,0,0,2.469895,19.585107,27.226714,33.618423,48.868213,25.908186,9.911899,{'observed_positive_with_source_limits': 456},456,False
1,osm_major_road_ratio,data/raw/taiwan-250101-free.shp.zip,ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab,six-class narrow research major line representation,major line union km in same primary boundary policy,osm_road_length_km,EPSG:4326,EPSG:3826,A metadata,包內README明示2025-01-01T21:21:00Z快照；非2024／2025逐年獨立路網,source coverage not certified as a full census; frozen original villages preserved,Ready with explicit assumptions,priority external comparison candidate,"Unit1.7 six classes, not official road hierarchy or traffic share; no road denominator => NA",DE / DA / PM pending,456,0,238,0.000000,0.000000,0.000000,0.135074,0.488236,0.076480,0.106696,"{'observed_zero_not_confirmed_absent': 238, 'observed_positive_with_source_limits': 218}",456,False
2,observed_signal_location_density,data/raw/號誌路口經緯度一覽20230706.csv,4de42a369f6b6ba38df29ff2d357f8ad5aec7a237b8709edc431d77c3cfb29a7,unique exact coordinate locations in conservative A policy,"readable non-test, unique interior, all basic member districts recognized and matching locations",fixed_village_area_km2,EPSG:4326 working interpretation,EPSG:3826,B working convention,檔名20230706非每筆觀測／設置時間；實際快照時點仍未認證,source coverage not certified as a full census; frozen original villages preserved,Partially ready,conditional priority; DA review of A/B and LCS sensitivity,"not physical intersections; uncertain LCS meanings/date/completeness, duplicate and source district conflicts retained",DE / DA / PM pending,456,0,27,0.000000,8.674376,19.606408,33.635733,116.584077,22.441216,17.400425,"{'observed_positive_with_source_limits': 429, 'observed_zero_not_confirmed_absent': 23, 'locations_excluded_by_primary_quality_policy': 4}",456,False
3,crosswalk_area_ratio,data/raw/(交工處)行人穿越線圖資_202407121519.zip,a7f4241ea1ecccfb09050fb38f4d8537a7f5ec1d4134eb7b8929217601ba0493,six-layer crossing-marking polygon union,observed marking union area_m2,fixed_village_area_m2,EPSG:3826,EPSG:3826,A metadata,2024-07-12封裝標記，非全年有效或2025新增完整紀錄；RDDATE需排除1899占位,source coverage not certified as a full census; frozen original villages preserved,Partially ready,supplementary observed markings comparison,components not full facilities; source coverage and historical stock unknown,DE / DA / PM pending,456,0,0,0.000019,0.002621,0.006484,0.009135,0.019291,0.006334,0.004110,{'observed_positive_with_source_limits': 456},456,False
4,median_annotated_road_width,data/raw/路寬檔_提供.zip,a92d7b7225509489181606158d1d9cbe3b1a4c83b120ee12d577460109852d70,"unique exact geometry + width annotation, assigned by midpoint",median of unique-midpoint valid positive annotation widths,median operator over sample count,EPSG:3826,EPSG:3826,A metadata,ZIP內資料修改2023-12-28，XML製作2021；非2024／2025完整年度實測,source coverage not certified as a full census; frozen original villages preserved,Partially ready,supplementary planned annotation sample description,not measured road wid

**Blocked未建立候選**

,feature,readiness_status,key_limitations,formal_numeric_column_created,recommended_analytical_use
0,intersection_density,Blocked,缺完整節點／轉向／立體拓樸，平面交叉不當實體路口,False,do not use
1,complete_topological_intersection_count,Blocked,免費線圖層缺完整節點與立體道路拓樸,False,do not use
2,complete_crosswalk_count,Blocked,一Polygon不是完整設施,False,do not use
3,crosswalk_length_density,Blocked,標線面構件不提供可靠穿越長度,False,do not use
4,length_weighted_road_width,Blocked,尺寸線不是縱向道路權重,False,do not use
5,narrow_road_length_ratio,Blocked,無可靠縱向分母與窄寬門檻,False,do not use
6,wide_road_length_ratio,Blocked,無可靠縱向分母與窄寬門檻,False,do not use
7,marked_sidewalk_density,Blocked,混合標線未驗證人行道子集,False,do not use
8,complete_signalized_intersection_count,Blocked,位置紀錄未認證實體路口單位,False,do not use
9,marked_sidewalk_area_ratio,Blocked,混合標線未驗證人行道子集，不能計算正式面積或覆蓋率,False,do not use


{'2024_fit_only': 'all future feature choice, low-count/smoothing fitted parameters, scaler, imputer, PCA and cluster centres must fit2024 alone; apply frozen transforms to2025', 'pooled_for_cross_year_fit': False, 'static_B2_B3_are_annual_measurements': False, 'MRT2026_is_pre_accident': False, 'models_fitted_in_unit27': False, 'same_village_years_independent': False}
none created: no downstream operation currently requires strict0–1. Original values/flags retained; do not globally clip. Future derivatives require justified source-based tolerance and separate names.
2025-06龍福+1075、幸福-1066；既有官方搬遷與暫遷戶籍制度證據僅支持背景，與人口跳動直接關係未證實；保留原值及行政里，Chapter3/5按需求審查兩里敏感度，不把設籍人口當交通暴露。


### Assertions

In [6]:
assert set(features.loc[features.allowed_for_main_clustering,'column_name'])==set(u.A2)
assert features.column_name.is_unique and len(features)==494
assert u.validate_registry(registry,frames,{p:contracts[p]['columns'] for p in frames})
assert len(registry['chapter2_inventory'])==17 and registry['chapter3_started'] is False
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h
u.verify(PROJECT_ROOT,INPUTS)
result={'status':'passed','independent_saved_file_read':True,'feature_entries':len(features),'role_counts':registry['role_counts'],'final_schema_variants_verified':True,'registry_sha256':sha(PROJECT_ROOT/u.REGISTRY),'producer_sha256':sha(PROJECT_ROOT/u.PRODUCER),'human_acceptance':'DE handoff / DA acceptance / PM review pending'}
(PROJECT_ROOT/'data/temp/unit27_integration/documentation_registry_summary.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2));print('本地技術交付；完成停止Unit2.7。')


{
  "status": "passed",
  "independent_saved_file_read": true,
  "feature_entries": 494,
  "role_counts": {
    "identifier": 13,
    "numerator_denominator": 195,
    "metadata": 40,
    "burden_outcome": 9,
    "quality_indicator": 174,
    "clustering_candidate": 9,
    "sensitivity": 39,
    "core_external": 3,
    "supplementary_external": 10,
    "priority_external": 2
  },
  "final_schema_variants_verified": true,
  "registry_sha256": "57c36e5572be721de86dabe992be2a0e0828893bab493c6ae104884cd062309c",
  "producer_sha256": "b0083fc4b44c5193f95b09dc669fac9cf9744e1bc1364760d860872bc7a452ba",
  "human_acceptance": "DE handoff / DA acceptance / PM review pending"
}
本地技術交付；完成停止Unit2.7。
